# Caspase-Active Compounds: UMAP + DBSCAN Clustering

**Focus:** Only cases where **caspase-3/7 is ACTIVE** across all comparison tables.  
Both-inactive excluded throughout. G4 (Mito vs MitoViab) excluded — it has no caspase dimension.

The 144+17 logic from pairwise Casp vs Mito extends to every table:
- Take all cases where caspase IS active
- Color dark → light = more co-active assays → fewer co-active assays
- Shape = which comparison group

| Shape | Group | Caspase-active cases kept |
|-------|-------|---------------------------|
| ● circle   | G1 | 3-way Casp+Mito+MitoViab: C1(63), C2(43), C4(17) |
| ▲ triangle | G2 | 3-way Casp+Mito+CaspViab: C1(101), C2(11), C3(6), C4(5) |
| ■ square   | G3 | Pairwise Casp vs Mito: C1(144) + C2(17) = **161** |
| ✚ plus     | G5 | Pairwise CHO-K1: C1(107) |
| ★ star     | G6 | Pairwise HepG2: C1(151) + C2(24) = 175 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import umap
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from collections import Counter, defaultdict
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})
print('Libraries loaded.')

## Step 1 — Build CID → SMILES mapping

In [ ]:
BASE = '/Users/yaminichitikela/Documents/casepase/Casepase-activation/cleaned_datasets/'

smiles_map = {}
for fname in ['combined_hepg2_casepase.csv', 'AID_720635_cleaned.csv',
              'AID_1346981_cleaned.csv', 'AID_1346980_cleaned.csv',
              'AID_1346979_cleaned.csv', 'AID_1346978_cleaned.csv',
              'AID_720634_cleaned.csv']:
    try:
        df = pd.read_csv(BASE + fname,
                         usecols=['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES'],
                         low_memory=False)
        df = df.dropna(subset=['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES'])
        df = df.drop_duplicates('PUBCHEM_CID')
        df['PUBCHEM_CID'] = df['PUBCHEM_CID'].astype(int)
        smiles_map.update(dict(zip(df['PUBCHEM_CID'], df['PUBCHEM_EXT_DATASOURCE_SMILES'])))
    except Exception as e:
        print(f'  Warning [{fname}]: {e}')

print(f'Total CID -> SMILES: {len(smiles_map)}')

## Step 2 — Load activity outcome sets

In [ ]:
def cids_with(path, outcome):
    df = pd.read_csv(path,
                     usecols=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME'],
                     low_memory=False)
    df = df.dropna(subset=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME'])
    df['PUBCHEM_CID'] = df['PUBCHEM_CID'].astype(int)
    pri = {'Active': 2, 'Inconclusive': 1, 'Inactive': 0}
    df['_p'] = df['PUBCHEM_ACTIVITY_OUTCOME'].map(pri).fillna(-1)
    df = df.sort_values('_p', ascending=False).drop_duplicates('PUBCHEM_CID')
    return set(df[df['PUBCHEM_ACTIVITY_OUTCOME'] == outcome]['PUBCHEM_CID'])

# HepG2 combined caspase (combined_hepg2_casepase.csv)
casp_A   = cids_with(BASE + 'combined_hepg2_casepase.csv', 'Active')
casp_I   = cids_with(BASE + 'combined_hepg2_casepase.csv', 'Inactive')

# Mito toxicity HepG2 (AID 720635)
mito_A   = cids_with(BASE + 'AID_720635_cleaned.csv', 'Active')
mito_I   = cids_with(BASE + 'AID_720635_cleaned.csv', 'Inactive')

# Mito viability HepG2 (AID 720634)
mviab_A  = cids_with(BASE + 'AID_720634_cleaned.csv', 'Active')
mviab_I  = cids_with(BASE + 'AID_720634_cleaned.csv', 'Inactive')

# Caspase viability HepG2 (AID 1346981)
cviab_A  = cids_with(BASE + 'AID_1346981_cleaned.csv', 'Active')
cviab_I  = cids_with(BASE + 'AID_1346981_cleaned.csv', 'Inactive')

# CHO-K1 caspase (AID 1346980)
cho_ca_A = cids_with(BASE + 'AID_1346980_cleaned.csv', 'Active')
cho_ca_I = cids_with(BASE + 'AID_1346980_cleaned.csv', 'Inactive')

# CHO-K1 viability (AID 1346979)
cho_vi_A = cids_with(BASE + 'AID_1346979_cleaned.csv', 'Active')
cho_vi_I = cids_with(BASE + 'AID_1346979_cleaned.csv', 'Inactive')

# HepG2 caspase activation (AID 1346978)
hca_A    = cids_with(BASE + 'AID_1346978_cleaned.csv', 'Active')
hca_I    = cids_with(BASE + 'AID_1346978_cleaned.csv', 'Inactive')

print('Activity sets loaded.')

## Step 3 — Define caspase-active-only categories

**Rule:** only keep cases where caspase IS active.  
- G1: Three-way cases C1, C2, C4 (C3=0 compounds)  
- G2: Three-way cases C1, C2, C3, C4  
- G3: Pairwise cases C1 (144) + C2 (17) = 161  
- G5: Pairwise CHO-K1 C1 (107)  
- G6: Pairwise HepG2 C1 (151) + C2 (24) = 175  
- G4 excluded (no caspase dimension in that table)

In [ ]:
# (key, group, case_number, label, cid_set)
# case_number controls shade: 1 = darkest (most co-active) -> highest = lightest
CATEGORIES = [
    # G1: Three-way Casp+Mito+MitoViab — CASPASE ACTIVE cases only
    ('G1_C1','G1',1,'Casp+Mito+MitoViab all active (n=63)',     casp_A & mito_A & mviab_A),
    ('G1_C2','G1',2,'Casp+Mito active, MitoViab inactive (n=43)', casp_A & mito_A & mviab_I),
    ('G1_C4','G1',3,'Casp active, Mito+MitoViab inactive (n=17)', casp_A & mito_I & mviab_I),

    # G2: Three-way Casp+Mito+CaspViab — CASPASE ACTIVE cases only
    ('G2_C1','G2',1,'Casp+Mito+CaspViab all active (n=101)',    casp_A & mito_A & cviab_A),
    ('G2_C2','G2',2,'Casp+Mito active, CaspViab inactive (n=11)', casp_A & mito_A & cviab_I),
    ('G2_C3','G2',3,'Casp+CaspViab active, Mito inactive (n=6)', casp_A & mito_I & cviab_A),
    ('G2_C4','G2',4,'Casp active, Mito+CaspViab inactive (n=5)', casp_A & mito_I & cviab_I),

    # G3: Pairwise Casp vs Mito — CASPASE ACTIVE: 144 + 17 = 161
    ('G3_C1','G3',1,'Casp+Mito both active (n=144)',            casp_A & mito_A),
    ('G3_C2','G3',2,'Casp active, Mito inactive (n=17)',        casp_A & mito_I),

    # G5: Pairwise CHO-K1 — CASPASE ACTIVE cases only
    ('G5_C1','G5',1,'CHO-K1: Casp+Viab both active (n=107)',   cho_ca_A & cho_vi_A),
    ('G5_C2','G5',2,'CHO-K1: Casp active, Viab inactive (n=0)', cho_ca_A & cho_vi_I),

    # G6: Pairwise HepG2 Casp vs Viability — CASPASE ACTIVE: 151 + 24 = 175
    ('G6_C1','G6',1,'HepG2: Casp+Viab both active (n=151)',    hca_A & cviab_A),
    ('G6_C2','G6',2,'HepG2: Casp active, Viab inactive (n=24)', hca_A & cviab_I),
]

print(f'{'Key':8s}  {'n actual':>9s}  Label')
print('-' * 70)
for key, grp, case, lbl, cids in CATEGORIES:
    print(f'{key:8s}  {len(cids):>9d}  {lbl}')
print('-' * 70)
total_unique = len(set().union(*[c for *_, c in CATEGORIES]))
print(f'Total unique caspase-active CIDs: {total_unique}')

## Step 4 — Assign primary label (G1 > G2 > G3 > G5 > G6)

In [ ]:
cid_meta = {}  # CID -> (key, grp, case, lbl)
for key, grp, case, lbl, cids in CATEGORIES:
    for cid in cids:
        if cid not in cid_meta:
            cid_meta[cid] = (key, grp, case, lbl)

cnt = Counter(v[0] for v in cid_meta.values())
print(f'Total caspase-active CIDs assigned: {len(cid_meta)}')
print('\nPrimary label distribution:')
for key, grp, case, lbl, _ in CATEGORIES:
    if cnt[key] > 0:
        print(f'  {key:8s}  n={cnt[key]:4d}')

## Step 5 — Validate SMILES and compute Morgan fingerprints (radius=2, 2048-bit)

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

valid_cids, valid_keys, valid_grps, valid_cases = [], [], [], []
fps_list = []

for cid, (key, grp, case, lbl) in cid_meta.items():
    smi = smiles_map.get(cid)
    if not smi or pd.isna(smi):
        continue
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    valid_cids.append(cid)
    valid_keys.append(key)
    valid_grps.append(grp)
    valid_cases.append(case)
    fps_list.append(gen.GetFingerprintAsNumPy(mol))

fps         = np.array(fps_list, dtype=np.uint8)
valid_keys  = np.array(valid_keys)
valid_grps  = np.array(valid_grps)
valid_cases = np.array(valid_cases)

print(f'Valid caspase-active compounds: {len(valid_cids)}')
print(f'Fingerprint matrix: {fps.shape}')

## Step 6 — UMAP (Jaccard distance on binary Morgan fingerprints)

Smaller N → use `n_neighbors=min(15, N-1)` to avoid UMAP errors.

In [ ]:
N = len(valid_cids)
n_neighbors = min(15, N - 1)

print(f'Running UMAP on {N} caspase-active compounds (n_neighbors={n_neighbors})...')
reducer = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=0.1,
    metric='jaccard',
    random_state=42,
    n_jobs=1
)
embedding = reducer.fit_transform(fps)
print(f'Embedding shape: {embedding.shape}')

## Step 7 — k-distance elbow plot to choose DBSCAN eps

In [ ]:
K = min(5, N - 1)  # smaller dataset -> smaller k

nbrs   = NearestNeighbors(n_neighbors=K, metric='euclidean').fit(embedding)
dist, _ = nbrs.kneighbors(embedding)
k_dist  = np.sort(dist[:, K - 1])  # ascending

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_dist, color='steelblue', linewidth=1.5)
ax.set_xlabel('Points sorted by k-NN distance (ascending)', fontsize=11)
ax.set_ylabel(f'{K}-NN distance in UMAP space', fontsize=11)
ax.set_title(f'k-Distance Graph (k={K}) — elbow = recommended eps', fontsize=12)
ax.grid(True, alpha=0.3)
for eps_c in [0.2, 0.3, 0.5, 0.7]:
    ax.axhline(eps_c, color='salmon', linestyle='--', lw=0.9)
    ax.text(N * 0.02, eps_c + 0.015, f'eps={eps_c}', fontsize=8, color='salmon')
plt.tight_layout()
plt.savefig(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_kdist_elbow.png',
    dpi=130, bbox_inches='tight'
)
plt.show()
print('Inspect the elbow, then set EPS in Step 8.')

## Step 8 — Fit DBSCAN + silhouette score

In [ ]:
EPS         = 0.3   # <-- set from elbow plot above
MIN_SAMPLES = 5

db        = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
db_labels = db.fit_predict(embedding)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
print(f'eps={EPS}, min_samples={MIN_SAMPLES}')
print(f'Clusters : {n_clusters}')
print(f'Noise    : {n_noise}  ({100 * n_noise / len(db_labels):.1f}%)')

ok = db_labels >= 0
if ok.sum() > 1 and n_clusters > 1:
    sil = silhouette_score(embedding[ok], db_labels[ok])
    print(f'Silhouette score (non-noise): {sil:.3f}  [-1 to +1, higher = better-defined clusters]')

## Step 9 — Marker shapes + color palette (dark → light within each group)

In [ ]:
# Shape = group (same as pairwise_analysis_26.ipynb for consistency)
MARKERS = {'G1': 'o', 'G2': '^', 'G3': 's', 'G5': 'P', 'G6': '*'}
MARKER_NAMES = {
    'G1': 'circle  ●', 'G2': 'triangle ▲', 'G3': 'square  ■',
    'G5': 'plus    ✚', 'G6': 'star    ★'
}
GROUP_LABELS = {
    'G1': 'G1: 3-way Casp+Mito+MitoViab  (casp-active only)',
    'G2': 'G2: 3-way Casp+Mito+CaspViab  (casp-active only)',
    'G3': 'G3: Pairwise Casp vs Mito      (144+17=161)',
    'G5': 'G5: Pairwise CHO-K1 Casp vs Viab (107)',
    'G6': 'G6: Pairwise HepG2 Casp vs Viab  (151+24=175)',
}
GROUP_CMAPS = {
    'G1': plt.cm.Blues, 'G2': plt.cm.Reds,
    'G3': plt.cm.Greens, 'G5': plt.cm.Oranges, 'G6': plt.cm.GnBu
}

grp_max = defaultdict(int)
for _, grp, case, *_ in CATEGORIES:
    grp_max[grp] = max(grp_max[grp], case)

CAT_COLOR = {}
for key, grp, case, *_ in CATEGORIES:
    nmax  = grp_max[grp]
    shade = 0.88 - (case - 1) / max(nmax - 1, 1) * 0.55
    CAT_COLOR[key] = GROUP_CMAPS[grp](shade)

print('Shape and color palette defined:')
for g, m in MARKERS.items():
    print(f'  {MARKER_NAMES[g]:16s}  {GROUP_LABELS[g]}')

## Step 10 — Figure A: UMAP by caspase-active category (shape + color)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

for key, grp, case, lbl, _ in CATEGORIES:
    mask = valid_keys == key
    n    = mask.sum()
    if n == 0:
        continue
    ax.scatter(
        embedding[mask, 0], embedding[mask, 1],
        marker=MARKERS[grp], color=CAT_COLOR[key],
        s=60 if grp in ('G1', 'G2') else 45,
        alpha=0.85, linewidths=0.5, edgecolors='grey',
        zorder=4 if grp in ('G1', 'G2') else 2,
        label=f'{key} (n={n})',
    )

ax.set_title(
    'UMAP — Caspase-Active Compounds Only\n'
    'Shape = comparison group  |  Color = case (dark: all assays active → light: caspase only)',
    fontsize=12
)
ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
ax.grid(True, alpha=0.15)

# Legend A: shapes
sh = [
    mlines.Line2D([0],[0], marker=MARKERS[g], color='dimgrey',
                  linestyle='None', markersize=11,
                  label=f'{MARKER_NAMES[g]}  {GROUP_LABELS[g]}')
    for g in MARKERS
]
leg1 = ax.legend(handles=sh, title='SHAPE = Group',
                 loc='lower left', fontsize=8, title_fontsize=9,
                 framealpha=0.92)
ax.add_artist(leg1)

# Legend B: colors
ch = [
    mlines.Line2D([0],[0], marker=MARKERS[grp], color=CAT_COLOR[key],
                  linestyle='None', markersize=9,
                  label=f'{key}  {lbl}')
    for key, grp, case, lbl, _ in CATEGORIES
    if (valid_keys == key).sum() > 0
]
ax.legend(handles=ch, title='COLOR = Case (dark → light)',
          bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=8, title_fontsize=9, framealpha=0.92)

plt.tight_layout()
plt.savefig(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_umap_categories.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: caspase_active_umap_categories.png')

## Step 11 — Figure B: UMAP by DBSCAN cluster (shape=group, color=cluster)

In [ ]:
cmap_db     = plt.cm.tab20
cluster_ids = sorted(set(db_labels))

fig, ax = plt.subplots(figsize=(13, 9))

for cl in cluster_ids:
    color = 'lightgrey' if cl == -1 else cmap_db(cl % 20)
    for grp, mkr in MARKERS.items():
        sub = (db_labels == cl) & (valid_grps == grp)
        if sub.sum() == 0:
            continue
        ax.scatter(
            embedding[sub, 0], embedding[sub, 1],
            marker=mkr, color=color,
            s=60 if grp in ('G1','G2') else 45,
            alpha=0.80, linewidths=0.5,
            edgecolors='grey' if cl != -1 else 'none',
            label=(f'C{cl} (n={(db_labels==cl).sum()})' if cl != -1 else 'Noise')
                  if grp == list(MARKERS.keys())[0] else '_nolegend_',
        )

ax.set_title(
    f'UMAP — DBSCAN Clusters on Caspase-Active Compounds  (eps={EPS}, min_s={MIN_SAMPLES})\n'
    f'{n_clusters} clusters  |  {n_noise} noise  |  Shape = group  |  Color = cluster',
    fontsize=12
)
ax.set_xlabel('UMAP 1', fontsize=11)
ax.set_ylabel('UMAP 2', fontsize=11)
ax.grid(True, alpha=0.15)

sh = [
    mlines.Line2D([0],[0], marker=MARKERS[g], color='dimgrey',
                  linestyle='None', markersize=11, label=GROUP_LABELS[g])
    for g in MARKERS
]
leg1 = ax.legend(handles=sh, title='SHAPE = Group',
                 loc='lower left', fontsize=8, title_fontsize=9, framealpha=0.92)
ax.add_artist(leg1)

cl_h = [
    mpatches.Patch(
        color='lightgrey' if cl == -1 else cmap_db(cl % 20),
        label='Noise' if cl == -1 else f'C{cl} (n={(db_labels==cl).sum()})'
    )
    for cl in cluster_ids
]
ax.legend(handles=cl_h, title='COLOR = Cluster',
          bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=8, title_fontsize=9, framealpha=0.92)

plt.tight_layout()
plt.savefig(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_umap_dbscan.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: caspase_active_umap_dbscan.png')

## Step 12 — Figure C: Side-by-side panel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# Left: categories
ax1 = axes[0]
for key, grp, case, lbl, _ in CATEGORIES:
    mask = valid_keys == key
    if mask.sum() == 0:
        continue
    ax1.scatter(embedding[mask,0], embedding[mask,1],
                marker=MARKERS[grp], color=CAT_COLOR[key],
                s=50, alpha=0.83, linewidths=0.4, edgecolors='grey',
                zorder=4 if grp in ('G1','G2') else 2)
ax1.set_title('(A) Caspase-Active Categories\nShape=group | Color=case (dark→light)', fontsize=11)
ax1.set_xlabel('UMAP 1'); ax1.set_ylabel('UMAP 2')
ax1.grid(True, alpha=0.15)
sh = [mlines.Line2D([0],[0], marker=MARKERS[g], color='dimgrey',
                    linestyle='None', markersize=10, label=g)
      for g in MARKERS]
ax1.legend(handles=sh, title='Shape', loc='lower left',
           fontsize=8, title_fontsize=9, framealpha=0.90)

# Right: DBSCAN
ax2 = axes[1]
for cl in cluster_ids:
    color = 'lightgrey' if cl == -1 else cmap_db(cl % 20)
    for grp, mkr in MARKERS.items():
        sub = (db_labels == cl) & (valid_grps == grp)
        if sub.sum() == 0:
            continue
        ax2.scatter(embedding[sub,0], embedding[sub,1],
                    marker=mkr, color=color,
                    s=50, alpha=0.80, linewidths=0.4,
                    edgecolors='grey' if cl != -1 else 'none')
ax2.set_title(f'(B) DBSCAN Clusters (eps={EPS}, min_s={MIN_SAMPLES})\n'
              f'{n_clusters} clusters | {n_noise} noise | Shape=group | Color=cluster', fontsize=11)
ax2.set_xlabel('UMAP 1'); ax2.set_ylabel('UMAP 2')
ax2.grid(True, alpha=0.15)
cl_h = [mpatches.Patch(color='lightgrey' if cl==-1 else cmap_db(cl%20),
                        label='Noise' if cl==-1 else f'C{cl} (n={(db_labels==cl).sum()})')
        for cl in cluster_ids]
ax2.legend(handles=cl_h, title='Cluster', bbox_to_anchor=(1.01,1), loc='upper left',
           fontsize=7.5, title_fontsize=8, framealpha=0.9)

plt.suptitle(
    'Caspase-Active Compound Chemical Space (HepG2 + CHO-K1)\n'
    'Morgan FP (r=2, 2048-bit) → UMAP (Jaccard) → DBSCAN',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_panel.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: caspase_active_panel.png')

## Step 13 — Cluster composition and enrichment table

In [ ]:
df_res = pd.DataFrame({
    'CID':    valid_cids,
    'Key':    valid_keys,
    'Group':  valid_grps,
    'Case':   valid_cases,
    'DBSCAN': db_labels,
    'UMAP1':  embedding[:, 0],
    'UMAP2':  embedding[:, 1],
})

# Cluster × Group composition
comp = df_res.groupby(['DBSCAN','Group']).size().unstack(fill_value=0)
comp.index = ['Noise' if i == -1 else f'C{i}' for i in comp.index]
print('Cluster × Group composition:')
print(comp.to_string())

# Enrichment
lbl_map = {key: lbl for key, _, _, lbl, _ in CATEGORIES}
print(f'\n{'Cluster':10s}  {'n':>5s}  {'Dom.Key':8s}  {'%':>5s}  Interpretation')
print('─' * 72)
for cl in sorted(set(db_labels)):
    sub = df_res[df_res['DBSCAN'] == cl]
    if len(sub) < 3:
        continue
    top     = sub['Key'].value_counts()
    dom_key = top.index[0]
    frac    = top.iloc[0] / len(sub)
    flag    = '  *** ENRICHED' if frac > 0.6 else ''
    cl_nm   = 'Noise' if cl == -1 else f'Cluster {cl}'
    print(f'{cl_nm:10s}  {len(sub):>5d}  {dom_key:8s}  {frac:>4.0%}  {lbl_map.get(dom_key,"")[:36]}{flag}')

# Cross-cell-line overlap: HepG2 vs CHO-K1 caspase-active
hepg2_cids = set(df_res[df_res['Group'].isin(['G1','G2','G3','G6'])]['CID'])
chok1_cids = set(df_res[df_res['Group'] == 'G5']['CID'])
overlap     = hepg2_cids & chok1_cids
print(f'\nCross-cell-line overlap (caspase-active in BOTH HepG2 and CHO-K1): {len(overlap)} compounds')

df_res.to_csv(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_cluster_assignments.csv',
    index=False
)
print('Saved: caspase_active_cluster_assignments.csv')

## Step 14 — Legend card

In [ ]:
present = [(key, grp, case, lbl, CAT_COLOR[key])
           for key, grp, case, lbl, cids in CATEGORIES
           if len(cids) > 0]

fig, ax = plt.subplots(figsize=(13, max(5, len(present) * 0.45)))
ax.set_xlim(0, 13); ax.set_ylim(-0.5, len(present) + 2); ax.axis('off')

for x, txt in zip([0.3, 0.9, 2.5, 4.5],
                  ['Shape', 'Key', 'Color', 'Case label']):
    ax.text(x, len(present) + 1.2, txt, fontsize=9, fontweight='bold', color='#333')

for i, (key, grp, case, lbl, color) in enumerate(present):
    y = len(present) - i
    ax.plot(0.55, y, marker=MARKERS[grp], color='dimgrey', markersize=13, linestyle='None')
    ax.text(0.9, y, key, va='center', fontsize=9, fontweight='bold')
    ax.add_patch(plt.Rectangle((2.4, y - 0.35), 1.6, 0.7,
                               color=color, ec='#aaa', lw=0.5))
    ax.text(4.3, y, lbl, va='center', fontsize=8.5)

ax.set_title(
    'Legend Card — Caspase-Active Compounds Only\n'
    'Shape = group  |  Color = case (dark: all assays active → light: caspase only)',
    fontsize=10, pad=14
)
plt.tight_layout()
plt.savefig(
    '/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/caspase_active_legend_card.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Saved: caspase_active_legend_card.png')